# **Project Name -**  **PhonePe Pulse Data Analysis & Visualization**

**Project Type**    - EDA

**Contribution**    - Mangali Rahul

# **Project Summary -**

India's digital payment ecosystem has experienced unprecedented growth over the last few years, largely driven by the Unified Payments Interface (UPI). At the forefront of this revolution is PhonePe. To foster transparency and data-driven exploration, PhonePe released the "PhonePe Pulse" dataset—a massive repository of transaction, user, and insurance data spanning across states, districts, and pin codes from 2018 to the present. However, this raw data is stored in thousands of fragmented, nested JSON files, making it incredibly difficult to analyze in its native format.

This Capstone project focuses on building a complete end-to-end Data Engineering and Analytics pipeline to extract, transform, and visualize this massive dataset. The primary objective is to convert unstructured JSON data into actionable business intelligence through a highly interactive, public-facing web dashboard.

#### Methodology & Execution:

The project was executed in four distinct phases:

**1. Data Extraction & Transformation (ETL):** Using Python, a custom extraction script was built to systematically traverse the cloned PhonePe Pulse GitHub repository. The script parsed thousands of nested JSON files across three main categories (Aggregated, Map, and Top), handling data inconsistencies (such as missing device brands in later years) and transforming the unstructured data into clean, structured Pandas DataFrames.

**2. Database Management:** To ensure the data was highly queryable and scalable, a local SQLite database (phonepe_pulse.db) was engineered. The transformed data was successfully loaded into 9 distinct tables, creating a robust data warehouse for analytical querying.

**3. Exploratory Data Analysis (EDA):** Using SQL and Pandas, deep-dive analyses were performed to answer critical business case studies. This included identifying top-performing districts, uncovering the most popular smartphone brands used by the user base, and tracking the growth of transaction volumes and insurance policies over time.

**4. Interactive Dashboard Creation:** The final deliverable is a multi-page, interactive web application built with Streamlit and Plotly. The dashboard features 10 dynamic charts, including Treemaps, Funnel charts, and geographical filters, allowing non-technical stakeholders to effortlessly explore transaction hotspots and market penetration at a hyper-local level.

Ultimately, this project demonstrates a full-stack data science workflow, resulting in a powerful BI tool that uncovers deep insights into India's digital payment landscape.

# **GitHub Link -**

# **Problem Statement -**

The PhonePe Pulse dataset contains invaluable insights into India's digital payment habits, but it is locked within a complex, highly nested structure of thousands of individual JSON files scattered across multiple directories. For business leaders and analysts to make strategic decisions, this raw, fragmented data must be systematically harvested, cleaned, and centralized into a structured format.

Furthermore, the data must be visualized dynamically, as static reports cannot adequately capture the geographical nuances and temporal trends of a rapidly evolving digital market. The core problem is bridging the gap between raw, distributed JSON files and an accessible, interactive Business Intelligence dashboard.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import sqlite3
import plotly.express as px
import warnings

# Ignore warnings for a clean notebook presentation
warnings.filterwarnings('ignore')

### Dataset Loading

In [2]:
# Load Dataset from our local SQLite Database
conn = sqlite3.connect('phonepe_pulse.db')

# We will load our three primary transaction tables for this EDA
df_agg_trans = pd.read_sql_query("SELECT * FROM Aggregated_transaction", conn)
df_map_trans = pd.read_sql_query("SELECT * FROM Map_transaction", conn)
df_top_trans = pd.read_sql_query("SELECT * FROM Top_transaction", conn)

# Load user data
df_agg_user = pd.read_sql_query("SELECT * FROM Aggregated_user", conn)

print("✅ Data successfully loaded from phonepe_pulse.db!")

✅ Data successfully loaded from phonepe_pulse.db!


### Dataset First View

In [3]:
# Dataset First Look (Looking at the Aggregated Transactions)
df_agg_trans.head()

,State,Year,Quarter,Transaction_Type,Count,Amount
0,Andaman & Nicobar Islands,2018,1,Recharge & bill payments,4200,1.845307e+06
1,Andaman & Nicobar Islands,2018,1,Peer-to-peer payments,1871,1.213866e+07
2,Andaman & Nicobar Islands,2018,1,Merchant payments,298,4.525072e+05
3,Andaman & Nicobar Islands,2018,1,Financial Services,33,1.060142e+04
4,Andaman & Nicobar Islands,2018,1,Others,256,1.846899e+05


### Dataset Rows & Columns count

In [4]:
# Dataset Rows & Columns count
print(f"Aggregated Transactions Shape: {df_agg_trans.shape}")
print(f"Map Transactions Shape: {df_map_trans.shape}")
print(f"Top Transactions Shape: {df_top_trans.shape}")
print(f"Aggregated Users Shape: {df_agg_user.shape}")

Aggregated Transactions Shape: (5034, 6)
Map Transactions Shape: (20604, 6)
Top Transactions Shape: (18295, 7)
Aggregated Users Shape: (7128, 8)


### Dataset Information

In [5]:
# Dataset Info
df_agg_trans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5034 entries, 0 to 5033
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   State             5034 non-null   object 
 1   Year              5034 non-null   int64  
 2   Quarter           5034 non-null   int64  
 3   Transaction_Type  5034 non-null   object 
 4   Count             5034 non-null   int64  
 5   Amount            5034 non-null   float64
dtypes: float64(1), int64(3), object(2)
memory usage: 236.1+ KB


#### Duplicate Values

In [6]:
# Dataset Duplicate Value Count
duplicates = df_agg_trans.duplicated().sum()
print(f"Total duplicate rows in Aggregated Transactions: {duplicates}")

Total duplicate rows in Aggregated Transactions: 0


#### Missing Values/Null Values

In [7]:
# Missing Values/Null Values Count
print("Missing values in Aggregated Transactions:")
print(df_agg_trans.isnull().sum())

print("\nMissing values in Aggregated Users:")
print(df_agg_user.isnull().sum())

Missing values in Aggregated Transactions:
State               0
Year                0
Quarter             0
Transaction_Type    0
Count               0
Amount              0
dtype: int64

Missing values in Aggregated Users:
State                0
Year                 0
Quarter              0
Registered_Users     0
App_Opens            0
Device_Brand         0
Device_Count         0
Device_Percentage    0
dtype: int64


In [8]:
# Visualizing the missing values
# Since we handled missing data during our ETL pipeline (by replacing missing device brands with 'Unknown'), 
# our dataset is perfectly clean. We will visualize the 'Unknown' values instead of true nulls.
missing_devices = df_agg_user[df_agg_user['Device_Brand'] == 'Unknown'].shape[0]
total_devices = df_agg_user.shape[0]

fig = px.pie(
    names=['Known Devices', 'Unknown Devices (Handled)'], 
    values=[total_devices - missing_devices, missing_devices],
    title="Data Completeness: Handled Missing Device Data"
)
fig.show()

### What did you know about your dataset?

The dataset is derived from the official PhonePe Pulse repository. During the Data Engineering (ETL) phase, raw JSON files were parsed and consolidated into a robust SQLite database.
Because of our rigorous ETL pipeline, the dataset is highly structured and remarkably clean. We have zero true "Null" or "NaN" values. The only missing data originates from PhonePe's decision to stop reporting specific smartphone device brands after a certain year. Rather than dropping these rows and losing the user count data, we systematically replaced the missing strings with "Unknown" during the extraction phase to preserve the integrity of the total user metrics. The dataset spans multiple years and captures highly granular geographical data down to the district and pin-code levels.

## ***2. Understanding Your Variables***

In [9]:
# Dataset Columns
print("Aggregated Transaction Columns:")
print(df_agg_trans.columns.tolist())

print("\nAggregated User Columns:")
print(df_agg_user.columns.tolist())

Aggregated Transaction Columns:
['State', 'Year', 'Quarter', 'Transaction_Type', 'Count', 'Amount']

Aggregated User Columns:
['State', 'Year', 'Quarter', 'Registered_Users', 'App_Opens', 'Device_Brand', 'Device_Count', 'Device_Percentage']


In [10]:
# Dataset Describe
# Transposing the describe table (.T) makes it much easier to read when there are many columns!
print("Statistical Summary of Transactions:")
display(df_agg_trans.describe().T)

print("\nStatistical Summary of Users:")
display(df_agg_user.describe().T)

Statistical Summary of Transactions:


,count,mean,std,min,25%,50%,75%,max
Year,5034.0,2.021003e+03,1.999849e+00,2018.000000,2.019000e+03,2.021000e+03,2.023000e+03,2.024000e+03
Quarter,5034.0,2.500795e+00,1.118145e+00,1.000000,2.000000e+00,3.000000e+00,4.000000e+00,4.000000e+00
Count,5034.0,4.673902e+07,1.690968e+08,2.000000,5.808950e+04,5.158310e+05,1.166629e+07,2.393918e+09
Amount,5034.0,6.863772e+10,2.685200e+11,34.397212,3.993888e+07,4.394139e+08,1.102822e+10,3.095666e+12



Statistical Summary of Users:


,count,mean,std,min,25%,50%,75%,max
Year,7128.0,2.019838e+03,1.447545e+00,2018.0,2019.000000,2.020000e+03,2.021000e+03,2.024000e+03
Quarter,7128.0,2.424242e+00,1.137958e+00,1.0,1.000000,2.000000e+00,3.000000e+00,4.000000e+00
Registered_Users,7128.0,6.098127e+06,8.822868e+06,501.0,205686.000000,2.133804e+06,8.972299e+06,7.180780e+07
App_Opens,7128.0,1.860901e+08,4.340965e+08,0.0,0.000000,7.162734e+06,1.765590e+08,5.335171e+09
Device_Count,7128.0,4.854553e+05,1.057863e+06,0.0,5886.000000,6.577150e+04,4.160462e+05,1.134094e+07
Device_Percentage,7128.0,8.585859e-02,8.368347e-02,0.0,0.018321,4.942278e-02,1.387764e-01,4.783670e-01


### Check Unique Values for each variable.

In [11]:
# Check Unique Values for each variable in the Transactions dataset
print("Unique Values in Aggregated Transactions Table:")
for col in df_agg_trans.columns:
    unique_count = df_agg_trans[col].nunique()
    print(f"{col}: {unique_count} unique values")

print("\n----------------------------------\n")

print("Unique Values in Aggregated Users Table:")
for col in df_agg_user.columns:
    unique_count = df_agg_user[col].nunique()
    print(f"{col}: {unique_count} unique values")

Unique Values in Aggregated Transactions Table:
State: 36 unique values
Year: 7 unique values
Quarter: 4 unique values
Transaction_Type: 5 unique values
Count: 4966 unique values
Amount: 5034 unique values

----------------------------------

Unique Values in Aggregated Users Table:
State: 36 unique values
Year: 7 unique values
Quarter: 4 unique values
Registered_Users: 1008 unique values
App_Opens: 829 unique values
Device_Brand: 21 unique values
Device_Count: 6502 unique values
Device_Percentage: 6727 unique values


## **3. *Data Wrangling***

### Data Wrangling Code

In [13]:
# Write your code to make your dataset analysis ready.

# 1. Create a combined 'Location' column in Map data for easier plotting later
df_map_trans['State_District'] = df_map_trans['District'] + " (" + df_map_trans['State'] + ")"

# 2. Convert 'Year' and 'Quarter' to strings so Plotly treats them as categories (not continuous numbers)
df_agg_trans['Year_str'] = df_agg_trans['Year'].astype(str)
df_agg_trans['Quarter_str'] = 'Q' + df_agg_trans['Quarter'].astype(str)

# 3. Ensure 'Pincode' in Top Users is treated as a string (so it doesn't get formatted with commas like 500,072)
# We will query this directly from the DB to show how we wrangle specific subsets
df_top_users = pd.read_sql_query("SELECT * FROM Top_user WHERE Entity_Type = 'Pincodes'", conn)
df_top_users['Entity_Name'] = df_top_users['Entity_Name'].astype(str)
df_top_users['Location'] = df_top_users['Entity_Name'] + " (" + df_top_users['State'] + ")"

print("✅ Final data wrangling and type-conversions complete!")

✅ Final data wrangling and type-conversions complete!


### What all manipulations have you done and insights you found?

The bulk of the data manipulations for this project were performed upstream during the automated Data Engineering (ETL) phase. To make the dataset analysis-ready, the following transformations were executed:

1. JSON Flattening & Extraction: The raw PhonePe Pulse data was buried inside thousands of nested JSON files separated by Year and State. A custom Python script was written using os.walk() to iterate through the directory tree, parse the JSON files, and extract only the relevant business metrics.

2. Handling Missing Data Traps: PhonePe stopped recording specific smartphone brand data (e.g., Xiaomi, Apple) after a certain year, resulting in missing usersByDevice dictionaries. Instead of dropping these records (which would ruin our total user counts), a logical failsafe was coded to append these rows but label the device brand as "Unknown" with a 0.0 percentage.

3. String Standardization: Raw text data like "andaman-&-nicobar-islands" and "bengaluru urban district" were cleaned using string manipulation (.str.replace('-', ' ').str.title()) to look professional in visualizations.

4. Categorical Type Casting: In the notebook, temporal columns like Year and Quarter were explicitly converted to strings. This prevents our visualization library (Plotly) from treating years as continuous numbers (e.g., plotting "2018.5"). We also combined geographical columns (e.g., District + State) into single string identifiers to prevent overlapping names on our charts.

5. Database Aggregation: Rather than keeping the data in heavy Pandas DataFrames in memory, the manipulated data was pushed into a highly indexed SQLite relational database (phonepe_pulse.db), allowing us to query exact subsets of data in milliseconds.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1: Transaction Categories

In [14]:
# Chart - 1 visualization code
# Univariate Analysis: Transaction Categories
df_cat = df_agg_trans.groupby('Transaction_Type')['Amount'].sum().reset_index()

fig1 = px.pie(df_cat, values='Amount', names='Transaction_Type', 
             title='Total Transaction Amount by Payment Category',
             hole=0.4, color_discrete_sequence=px.colors.sequential.Purp)
fig1.update_traces(textinfo='percent+label')
fig1.show()

#### What is/are the insight(s) found from the chart?

Peer-to-peer (P2P) payments absolutely dominate the PhonePe ecosystem, accounting for roughly 77% of the total transaction amount. Merchant payments trail behind as the second largest category, while Recharges, Bill payments, and Financial Services make up a very small minority of the total monetary volume.

#### Chart - 2: Top Smartphone Brands

In [18]:
# Chart - 2 visualization code
# Univariate Analysis: Top Smartphone Brands
df_brands = df_agg_user[df_agg_user['Device_Brand'] != 'Unknown'].groupby('Device_Brand')['Device_Count'].sum().reset_index()
df_brands = df_brands.sort_values(by='Device_Count', ascending=False).head(10)

fig2 = px.bar(df_brands, x='Device_Brand', y='Device_Count', 
             title='Top 10 Smartphone Brands Used by PhonePe Users',
             labels={'Device_Brand': 'Smartphone Brand', 'Device_Count': 'Total Users'},
             color='Device_Count', color_continuous_scale=px.colors.sequential.Teal)
fig2.show()

#### What is/are the insight(s) found from the chart?

Xiaomi is by far the most popular smartphone brand among PhonePe users, followed closely by Samsung and Vivo. Apple (iOS) sits significantly further down the list, indicating that the vast majority of PhonePe's user base operates on the Android ecosystem, specifically on budget-friendly to mid-range devices.

#### Chart - 3: Transaction Amount Over Time

In [17]:
# Chart - 3 visualization code
# Bivariate Analysis: Year vs Transaction Amount
df_time = df_agg_trans.groupby('Year_str')['Amount'].sum().reset_index()

fig3 = px.line(df_time, x='Year_str', y='Amount', markers=True, 
              title='Growth of Total Transaction Amount Over Time',
              labels={'Year_str': 'Year', 'Amount': 'Total Amount (INR)'},
              color_discrete_sequence=['#8A2BE2'])
fig3.show()

#### What is/are the insight(s) found from the chart?

The chart reveals an aggressive, exponential growth trajectory in the total monetary value moving through PhonePe year over year. From 2018 to the present, the adoption of digital payments has skyrocketed without any signs of plateauing.

#### Chart - 4: Transaction Volume Over Time

In [19]:
# Chart - 4 visualization code
# Bivariate Analysis: Year vs Transaction Count (Volume)
df_time_vol = df_agg_trans.groupby('Year_str')['Count'].sum().reset_index()

fig4 = px.area(df_time_vol, x='Year_str', y='Count', markers=True,
              title='Growth of Total Transaction Volume (Count) Over Time',
              labels={'Year_str': 'Year', 'Count': 'Total Number of Transactions'},
              color_discrete_sequence=['#20B2AA'])
fig4.show()

#### What is/are the insight(s) found from the chart?

Similar to the transaction amount, the absolute number of transactions (the count) is growing exponentially. This implies that users aren't just sending larger amounts of money; they are using the app far more frequently for everyday, smaller transactions.

#### Chart - 5: Transaction Hotspots by District

In [20]:
# Chart - 5 visualization code
# Multivariate Analysis: State -> District -> Transaction Amount
# We will look at the top 20 performing districts overall
df_map_top = df_map_trans.groupby(['State', 'District'])['Amount'].sum().reset_index()
df_map_top = df_map_top.sort_values(by='Amount', ascending=False).head(20)

fig5 = px.treemap(df_map_top, path=[px.Constant("India"), 'State', 'District'], 
                 values='Amount', color='Amount',
                 title='Transaction Hotspots: Top 20 Districts in India',
                 color_continuous_scale='Viridis')
fig5.update_traces(root_color="lightgrey")
fig5.show()

#### What is/are the insight(s) found from the chart?

Bengaluru Urban (Karnataka) is the absolute dominant powerhouse of PhonePe transactions in India. It is followed by other major IT and metropolitan hubs like Hyderabad (Telangana), Pune (Maharashtra), and Central Delhi.

#### Chart - 6: Top States by Insurance Adoption

In [21]:
# Chart - 6 visualization code
# Bivariate Analysis: State vs Total Insurance Policies
df_ins = pd.read_sql_query("""
    SELECT State, SUM(Count) as Total_Policies 
    FROM Aggregated_insurance 
    GROUP BY State 
    ORDER BY Total_Policies DESC 
    LIMIT 15
""", conn)

fig6 = px.bar(df_ins, x='Total_Policies', y='State', orientation='h',
             title='Top 15 States by Insurance Policy Adoption',
             labels={'Total_Policies': 'Total Policies Sold', 'State': 'State'},
             color='Total_Policies', color_continuous_scale=px.colors.sequential.Sunset)
fig6.update_layout(yaxis={'categoryorder':'total ascending'})
fig6.show()

#### What is/are the insight(s) found from the chart?

Karnataka and Maharashtra are leading the charge in purchasing insurance policies through the PhonePe app. Southern and Western states generally show higher adoption rates compared to Northern and Eastern states.

#### Chart - 7: Top States by Total Transaction Amount

In [22]:
# Chart - 7 visualization code
# Bivariate Analysis: State vs Transaction Amount
df_state_amt = df_agg_trans.groupby('State')['Amount'].sum().reset_index()
df_state_amt = df_state_amt.sort_values(by='Amount', ascending=False).head(15)

fig7 = px.bar(df_state_amt, x='State', y='Amount', 
             title='Top 15 States by Total Transaction Amount',
             labels={'Amount': 'Total Amount (INR)', 'State': 'State'},
             color='Amount', color_continuous_scale='Magma')
fig7.update_layout(xaxis_tickangle=-45)
fig7.show()

#### What is/are the insight(s) found from the chart?

Telangana, Maharashtra, and Karnataka consistently output the highest monetary transaction volumes. There is a steep drop-off after the top 5 states, indicating that digital wealth transfer is heavily concentrated in specific geographical tech and commerce hubs.

#### Chart - 8: Most Engaged States by App Opens

In [26]:
# Chart - 8 visualization code
# Bivariate/Multivariate Analysis: State vs App Opens (Using a Bubble Chart)
df_opens = pd.read_sql_query("""
    SELECT State, SUM(App_Opens) as Total_Opens 
    FROM Map_user 
    GROUP BY State 
    ORDER BY Total_Opens DESC 
    LIMIT 20
""", conn)

fig8 = px.scatter(df_opens, x='State', y='Total_Opens', 
                 size='Total_Opens', color='Total_Opens',
                 title='Most Engaged States by Total App Opens',
                 color_continuous_scale='Plasma', size_max=40)
fig8.update_layout(xaxis_tickangle=-45)
fig8.show()

#### What is/are the insight(s) found from the chart?

The massive bubbles for states like Maharashtra and Uttar Pradesh indicate incredibly high daily active user engagement. Users in these states are opening the app habitually, likely multiple times a day.

#### Chart - 9: Top Pincodes for User Registrations

In [24]:
# Chart - 9 visualization code
# Univariate/Bivariate Analysis: Pincodes vs Registered Users
df_pins = pd.read_sql_query("""
    SELECT State, Entity_Name as Pincode, SUM(Registered_Users) as Total_Registrations 
    FROM Top_user 
    WHERE Entity_Type = 'Pincodes' 
    GROUP BY State, Pincode 
    ORDER BY Total_Registrations DESC 
    LIMIT 10
""", conn)

df_pins['Location'] = df_pins['Pincode'].astype(str) + " (" + df_pins['State'] + ")"

fig9 = px.funnel(df_pins, x='Total_Registrations', y='Location', 
                title='Top 10 Pincodes in India by Registered Users',
                color='Location')
fig9.update_layout(showlegend=False)
fig9.show()

#### What is/are the insight(s) found from the chart?

Pin code 201301 (Noida, Uttar Pradesh) has the highest concentration of registered users in the entire country, followed closely by specific pin codes in Delhi and Bengaluru.

#### Chart - 10: Top Districts for Insurance Policies

In [27]:
# Chart - 10 visualization code
# Bivariate Analysis: District vs Total Policies
df_dist_ins = pd.read_sql_query("""
    SELECT District, State, SUM(Count) as Total_Policies 
    FROM Map_insurance 
    GROUP BY District, State 
    ORDER BY Total_Policies DESC 
    LIMIT 10
""", conn)

df_dist_ins['Location'] = df_dist_ins['District'] + " (" + df_dist_ins['State'] + ")"

fig10 = px.bar(df_dist_ins, x='Total_Policies', y='Location', orientation='h',
              title='Top 10 Districts in India for Insurance Adoption',
              labels={'Total_Policies': 'Total Policies Sold', 'Location': 'District'},
              color='Total_Policies', color_continuous_scale='YlGnBu')
fig10.update_layout(yaxis={'categoryorder':'total ascending'})
fig10.show()

#### What is/are the insight(s) found from the chart?

Bengaluru Urban and Pune are not just transaction hubs; they are also the absolute leaders in buying insurance policies through the app. Urban, metropolitan districts heavily outpace rural districts in insurance adoption.

#### Chart - 11: Registered Users vs. App Opens (Engagement)

In [28]:
# Chart - 11 visualization code
# Bivariate/Correlation Analysis: Registered Users vs App Opens
df_engagement = pd.read_sql_query("""
    SELECT State, SUM(Registered_Users) as Total_Users, SUM(App_Opens) as Total_Opens 
    FROM Map_user 
    GROUP BY State
""", conn)

fig11 = px.scatter(df_engagement, x='Total_Users', y='Total_Opens', 
                  hover_name='State',
                  title='Correlation: Registered Users vs. App Opens by State',
                  labels={'Total_Users': 'Total Registered Users', 'Total_Opens': 'Total App Opens'},
                  color='Total_Opens', color_continuous_scale='Inferno')
fig11.show()

#### What is/are the insight(s) found from the chart?

There is a strong, positive, linear correlation. As the number of registered users in a state increases, the total app opens increase proportionally. Maharashtra and Uttar Pradesh are massive outliers in the top right, leading both in registrations and engagement.

#### Chart - 12: Transaction Volume by Quarter (Seasonality)

In [29]:
# Chart - 12 visualization code
# Univariate Analysis: Quarterly Transaction Trends
df_quarter = pd.read_sql_query("""
    SELECT Quarter, SUM(Count) as Total_Transactions 
    FROM Aggregated_transaction 
    GROUP BY Quarter
""", conn)

df_quarter['Quarter_Label'] = 'Q' + df_quarter['Quarter'].astype(str)

fig12 = px.bar(df_quarter, x='Quarter_Label', y='Total_Transactions',
              title='Transaction Volume by Quarter (Seasonality Analysis)',
              labels={'Quarter_Label': 'Financial Quarter', 'Total_Transactions': 'Total Number of Transactions'},
              color='Total_Transactions', color_continuous_scale='Blues')
fig12.show()

#### What is/are the insight(s) found from the chart?

Quarter 4 (Q4) typically sees the highest volume of transactions. This aligns perfectly with the Indian festive season (Diwali, Dussehra, New Year), where retail shopping, gifting, and spending are at their annual peak.

#### Chart - 13: Transaction Composition in Top States

In [30]:
# Chart - 13 visualization code
# Multivariate Analysis: State vs Transaction Amount grouped by Transaction Type
# Let's look at the composition of the top 5 states
top_5_states = df_state_amt.head(5)['State'].tolist()
df_comp = df_agg_trans[df_agg_trans['State'].isin(top_5_states)]
df_comp = df_comp.groupby(['State', 'Transaction_Type'])['Amount'].sum().reset_index()

fig13 = px.bar(df_comp, x='State', y='Amount', color='Transaction_Type',
              title='Transaction Type Composition in Top 5 States',
              labels={'Amount': 'Total Amount (INR)'},
              color_discrete_sequence=px.colors.qualitative.Pastel)
fig13.show()

#### What is/are the insight(s) found from the chart?

While Peer-to-Peer (P2P) payments are the largest segment across all top states, states like Maharashtra and Karnataka have a visibly thicker band for "Merchant payments" compared to others, indicating a higher density of offline retail adoption.

#### Chart - 14: Correlation Heatmap

In [31]:
# Chart - 14 - Correlation Heatmap visualization code
# First, let's merge our user engagement data with our transaction amount data to see how they correlate
df_macro = pd.merge(df_engagement, df_state_amt, on='State')

# Calculate the correlation matrix for the numerical columns
corr_matrix = df_macro[['Total_Users', 'Total_Opens', 'Amount']].corr()

fig14 = px.imshow(corr_matrix, text_auto=True, aspect="auto",
                 title='Correlation Heatmap: Users, Engagement, and Revenue',
                 color_continuous_scale='RdBu_r')
fig14.show()

#### Why did you pick the specific chart?

A Correlation Heatmap is the most statistically rigorous way to visualize relationships between multiple numerical variables simultaneously. It immediately shows whether metrics move together (positive correlation near 1) or opposite to each other.

#### What is/are the insight(s) found from the chart?

There is an incredibly high positive correlation (close to 0.95+) between Total_Users, Total_Opens, and Transaction Amount. This proves mathematically that acquiring a user leads directly to app engagement, which in turn leads directly to financial volume.

#### Chart - 15: Pair Plot

In [32]:
# Chart - 15 - Pair Plot visualization code
# Using the same macro dataset, we will plot the pairwise relationships
fig15 = px.scatter_matrix(df_macro, 
                         dimensions=['Total_Users', 'Total_Opens', 'Amount'],
                         hover_name='State',
                         title='Pair Plot: State-Level Macro Metrics',
                         color='Amount', color_continuous_scale='Viridis')
fig15.update_traces(diagonal_visible=False)
fig15.show()

#### Why did you pick the specific chart?

A Pair Plot (Scatter Matrix) complements the heatmap by showing the actual data distribution and scatter shape for every combination of our key metrics. It allows us to spot specific outliers that a simple correlation number might hide.

#### What is/are the insight(s) found from the chart?

The scatter plots show strict linear trends across all axes, confirming the heatmap's findings. However, we can also visibly identify the "whale" states (like Maharashtra and Telangana) pulling far away from the main cluster in the bottom left, highlighting the massive inequality in digital adoption between top-tier and lower-tier states.

## **5. Solution to Business Objective**

#### What do you suggest the client to achieve Business Objective ?

**Strategic Recommendations to Achieve the Business Objective:**

Based on the comprehensive Exploratory Data Analysis of the PhonePe Pulse dataset, the following data-driven strategies are recommended to maximize market penetration and revenue:

**1. Hyper-Local Merchant Expansion:** Geographical data (Treemaps and Funnel charts) proves that digital payment adoption is heavily skewed toward major tech hubs (Bengaluru, Pune, Hyderabad) and specific pin codes (e.g., Noida 201301). PhonePe should heavily deploy physical infrastructure (POS machines, SmartSpeakers) and offline sales agents in these saturated hubs to maintain dominance, while simultaneously identifying adjacent Tier-2 cities for early-adopter campaigns.

**2. App Optimization for Budget Androids:** The user data definitively shows that Xiaomi, Vivo, and Samsung dominate the PhonePe ecosystem, while Apple (iOS) holds a minimal share. The engineering team must prioritize lightweight, low-latency app updates optimized for mid-range Android devices. Heavy, memory-intensive UI updates risk alienating the core user base.

**3. Cross-Selling High-Margin Products:** While Peer-to-Peer (P2P) transfers drive the vast majority of transaction volume, they offer low revenue margins. However, the correlation data shows highly engaged users in states like Maharashtra and Karnataka. PhonePe must leverage this engagement by aggressively pushing high-margin financial products (Health/Bike Insurance, Gold purchases) via targeted push notifications specifically to users in these states.

**4. Predictive Server Scaling:** Time-series analysis reveals a consistent, massive spike in transaction volume during Quarter 4 (aligning with the Indian festive season). IT infrastructure must proactively pre-scale database capacity every Q4 to prevent server crashes, ensuring zero downtime when users rely on the app the most.

# **Business Case Study -**

## **Case Study 1 :**

In [2]:
import sqlite3
import pandas as pd
import plotly.express as px

# 1. Connect to your database
conn = sqlite3.connect('phonepe_pulse.db')

# 2. Write the corrected SQL Query for Case Study 1
query_1 = """
    SELECT 
        Transaction_Type, 
        SUM(Amount) as Total_Amount,
        SUM(Count) as Total_Transactions
    FROM Aggregated_transaction
    GROUP BY Transaction_Type
    ORDER BY Total_Amount DESC;
"""

# 3. Execute the query and load into a Pandas DataFrame
df_case1 = pd.read_sql_query(query_1, conn)

# Display the raw data table
print("Data Extracted for Case Study 1:")
display(df_case1)

# 4. Create an interactive Donut Chart using Plotly
fig = px.pie(
    df_case1, 
    values='Total_Amount', 
    names='Transaction_Type', 
    title='Case Study 1: Total Transaction Amount by Payment Category',
    hole=0.4,
    color_discrete_sequence=px.colors.sequential.Purp
)

fig.update_traces(textinfo='percent+label')
fig.show()

Data Extracted for Case Study 1:


,Transaction_Type,Total_Amount,Total_Transactions
0,Peer-to-peer payments,2.665274e+14,85032446653
1,Merchant payments,6.533988e+13,130238755487
2,Recharge & bill payments,1.333876e+13,19596755603
3,Others,1.742807e+11,262050188
4,Financial Services,1.420188e+11,154208943


The query executed perfectly, and the donut chart immediately gives us our first major business insight: Peer-to-peer payments absolutely dominate the platform, making up over 77% of the total money moved on PhonePe!

## **Case Study 2 :**

In [3]:
# 1. Write the SQL Query for Case Study 2
# Let's find the top 10 most popular smartphone brands used by PhonePe customers
query_2 = """
    SELECT 
        Device_Brand, 
        SUM(Device_Count) as Total_Users
    FROM Aggregated_user
    WHERE Device_Brand != 'Unknown'
    GROUP BY Device_Brand
    ORDER BY Total_Users DESC
    LIMIT 10;
"""

# 2. Execute the query and load into a Pandas DataFrame
df_case2 = pd.read_sql_query(query_2, conn)

# Display the raw data table
print("Data Extracted for Case Study 2:")
display(df_case2)

# 3. Create an interactive Bar Chart using Plotly
fig2 = px.bar(
    df_case2, 
    x='Device_Brand', 
    y='Total_Users',
    title='Case Study 2: Top 10 Smartphone Brands Used by PhonePe Users',
    labels={'Device_Brand': 'Smartphone Brand', 'Total_Users': 'Total Users'},
    color='Total_Users',
    color_continuous_scale=px.colors.sequential.Teal
)

fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

Data Extracted for Case Study 2:


,Device_Brand,Total_Users
0,Xiaomi,869562617
1,Samsung,671603711
2,Vivo,625415019
3,Oppo,420250245
4,Others,282950234
5,Realme,219973222
6,Apple,95947314
7,Motorola,73340734
8,OnePlus,63677211
9,Huawei,57129693


That is brilliant! Xiaomi takes the crown by a massive margin, with Samsung and Vivo trailing behind. This is a massive insight for PhonePe—if they are going to optimize their app's performance or push notifications, they better make sure it works flawlessly on Android, specifically Xiaomi devices!

## **Case Study 3 :**

In [4]:
# 1. Write the SQL Query for Case Study 3
# Let's find out which states have the highest insurance adoption
query_3 = """
    SELECT 
        State, 
        SUM(Count) as Total_Policies,
        SUM(Amount) as Total_Amount
    FROM Aggregated_insurance
    GROUP BY State
    ORDER BY Total_Policies DESC
    LIMIT 15;
"""

# 2. Execute the query and load into a Pandas DataFrame
df_case3 = pd.read_sql_query(query_3, conn)

# Display the raw data table
print("Data Extracted for Case Study 3:")
display(df_case3)

# 3. Create an interactive Horizontal Bar Chart using Plotly
fig3 = px.bar(
    df_case3, 
    x='Total_Policies', 
    y='State', 
    orientation='h',
    title='Case Study 3: Top 15 States by Insurance Adoption',
    labels={'Total_Policies': 'Number of Policies Sold', 'State': 'State'},
    color='Total_Amount',
    color_continuous_scale=px.colors.sequential.Sunset
)

# Sort the bars so the highest is at the top
fig3.update_layout(yaxis={'categoryorder':'total ascending'})
fig3.show()

Data Extracted for Case Study 3:


,State,Total_Policies,Total_Amount
0,Karnataka,1957404,2.743155e+09
1,Maharashtra,1815539,2.363129e+09
2,Tamil Nadu,1215269,1.555507e+09
3,Uttar Pradesh,1139153,1.740346e+09
4,Telangana,894342,1.171060e+09
5,West Bengal,839715,1.052463e+09
6,Kerala,824235,1.313719e+09
7,Andhra Pradesh,697769,8.122230e+08
8,Delhi,652514,8.153652e+08
9,Rajasthan,639684,9.596539e+08


Look at that! Karnataka and Maharashtra are absolutely dominating the insurance market on PhonePe. This tells the business exactly where to focus their next insurance marketing campaigns—or where they need to push harder to wake up the sleeping markets.

## **Case Study 4 :**

In [5]:
# 1. Write the SQL Query for Case Study 4
# Let's find the Top 20 districts in India driving the highest transaction amounts
query_4 = """
    SELECT 
        State, 
        District,
        SUM(Amount) as Total_Amount,
        SUM(Count) as Total_Transactions
    FROM Map_transaction
    GROUP BY State, District
    ORDER BY Total_Amount DESC
    LIMIT 20;
"""

# 2. Execute the query and load into a Pandas DataFrame
df_case4 = pd.read_sql_query(query_4, conn)

# Display the raw data table
print("Data Extracted for Case Study 4:")
display(df_case4)

# 3. Create an interactive Treemap using Plotly
fig4 = px.treemap(
    df_case4, 
    path=[px.Constant("India"), 'State', 'District'], 
    values='Total_Amount',
    title='Case Study 4: Top 20 Districts by Total Transaction Amount',
    color='Total_Amount',
    color_continuous_scale='Viridis',
    hover_data=['Total_Transactions']
)

fig4.update_traces(root_color="lightgrey")
fig4.update_layout(margin = dict(t=50, l=25, r=25, b=25))
fig4.show()

Data Extracted for Case Study 4:


,State,District,Total_Amount,Total_Transactions
0,Karnataka,Bengaluru Urban,1.993784e+13,17108133846
1,Telangana,Hyderabad,1.190694e+13,7701373666
2,Maharashtra,Pune,9.730218e+12,9369053544
3,Rajasthan,Jaipur,7.854092e+12,5396870406
4,Telangana,Rangareddy,7.155140e+12,5038415120
5,Telangana,Medchal Malkajgiri,5.758878e+12,4107321900
6,Andhra Pradesh,Visakhapatnam,4.198568e+12,2575752191
7,Andhra Pradesh,Guntur,3.174527e+12,1702223654
8,Andhra Pradesh,Krishna,3.142856e+12,1662350116
9,Bihar,Patna,3.110762e+12,1968265128


That Treemap looks incredible! And the insight is glaringly obvious: Bengaluru Urban in Karnataka is the absolute powerhouse of PhonePe transactions, dominating that massive yellow block on the screen, followed closely by Hyderabad in Telangana.

## **Case Study 5 :**

In [7]:
# 1. Write the SQL Query for Case Study 5
# Let's find the top 10 Pincodes with the highest number of registered users
query_5 = """
    SELECT 
        State, 
        Entity_Name as Pincode,
        SUM(Registered_Users) as Total_Registrations
    FROM Top_user
    WHERE Entity_Type = 'Pincodes'
    GROUP BY State, Pincode
    ORDER BY Total_Registrations DESC
    LIMIT 10;
"""

# 2. Execute the query and load into a Pandas DataFrame
df_case5 = pd.read_sql_query(query_5, conn)

# Display the raw data table
print("Data Extracted for Case Study 5:")
display(df_case5)

# 3. Create an interactive Funnel Chart using Plotly
# We will combine the Pincode and State for the labels so it's easy to read
df_case5['Location'] = df_case5['Pincode'].astype(str) + " (" + df_case5['State'] + ")"

fig5 = px.funnel(
    df_case5, 
    x='Total_Registrations', 
    y='Location',
    title='Case Study 5: Top 10 Pincodes for User Registrations',
    color='Location' # Color by the specific location instead of a continuous scale!
)

fig5.show()

Data Extracted for Case Study 5:


,State,Pincode,Total_Registrations
0,Uttar Pradesh,201301,16876987
1,Delhi,110059,14167569
2,Karnataka,560068,13002727
3,Maharashtra,421302,12366260
4,Telangana,500072,11771379
5,Delhi,110092,11205830
6,Haryana,122001,11127873
7,Delhi,110086,9749619
8,Haryana,121004,9675789
9,Uttar Pradesh,201009,9418660


The funnel chart worked perfectly, and it's fascinating to see Noida (201301 in UP) sitting at the very top with nearly 16.9 million registered users.

# **Conclusion -**

This Capstone project successfully bridged the gap between raw, unstructured data and actionable business intelligence.

By engineering a robust Python ETL pipeline, thousands of nested JSON files from the PhonePe Pulse GitHub repository were systematically extracted, cleaned, and loaded into a high-performance SQLite database. This centralized data warehouse enabled complex SQL querying, overcoming the initial challenge of fragmented data.

Through rigorous Exploratory Data Analysis (EDA), 15 distinct visualizations were generated, uncovering critical insights into India's digital payment ecosystem—from the absolute dominance of budget Android smartphones and the exponential year-over-year growth of transaction volumes, to the hyper-local identification of top-performing districts and pin codes. Furthermore, deploying this data into an interactive, multi-page Streamlit web dashboard provided a dynamic Business Intelligence tool capable of empowering stakeholders to make geographically targeted, data-driven decisions. Ultimately, this project demonstrates a complete, end-to-end Data Science workflow capable of transforming massive datasets into strategic market value.